# TDSC-ABUS challenge, team T1 (SZU): CLSNet, SEGNet, DETNet

## 1. Paper summary and reference

G. Luo, M. Xu, H. Chen, X. Liang, X. Tao, D. Ni, et al., *Tumor detection, segmentation and classification challenge on automated 3D breast ultrasound: the TDSC-ABUS challenge*, [arXiv:2501.15588](https://arxiv.org/abs/2501.15588) (2025).

The paper summarises a challenge, so each team gets a few sentences. We implement team T1, the winner (Sec. 3.2.1): three separate models for classification, segmentation and detection, evaluated with the challenge's metrics (Sec. 2.3). Other teams are tracked in their own issues.

In [ ]:
import sys
sys.path.insert(0, "..")
from pathlib import Path
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from abus_classification.challenge import PUBLISHED
from abus_classification.papers.luo2025_t1 import CLSNet, DETNet, SEGNet
from abus_classification.papers.luo2025_t1 import pipeline as p

RESULTS = Path("../results/luo2025_t1")
results = RESULTS / "luo2025_t1_results.csv"
print("stored results:", results.exists())

## 2. Implementation details

The paper names three model families and nothing else. Every other row is our assumption.

| Item | Paper | Ours | Status |
|---|---|---|---|
| CLSNet | 3D ResNet with softmax | torchvision `r3d_18`, one input channel, two classes, no pretraining | Depth **assumed** |
| CLSNet input | Not stated | 128 x 64 x 128 patch at 0.5 mm around DETNet's top box; trained around the true centre, moved up to 12 voxels | **Assumed** |
| SEGNet | U-Net | 3D U-Net, widths 16-256, depth 4, Dice + cross-entropy | Widths and loss **assumed** |
| SEGNet inference | Not stated | Sliding window at 50% overlap, threshold 0.5, largest component | **Assumed** |
| DETNet | Single-stage, anchor-free, after CPMNet | Heatmap (CornerNet focal loss), offset and size at stride 2 on whole 1 mm volumes, top 5 peaks | CPMNet's sampling is **not** copied |
| Voxels | Not stated | 0.5 mm isotropic with area averaging | **Assumed** |
| Optimizer | Not stated | Adam, lr 1e-3 (DETNet, SEGNet), 1e-4 (CLSNet), weight decay 1e-5 | **Assumed** |
| Augmentation | Not stated | Random flips; brightness and contrast for SEGNet and CLSNet | **Assumed** |
| Stopping | Not stated | Best epoch on Validation, lr halved on a plateau, patience 20-30, 1.5 h per training | **Our addition** |
| Training data | Challenge: Train + Validation available | Fit on Train (100), select on Validation (30) | **Deviation** from the shared 130 |
| Test input | Docker on unseen volumes | No ground truth used; a true-centre AUC is reported only as a reference | Follows |

## 3. Verification of the implementation

Unit tests are in `test/papers_tests/test_luo2025_t1.py`: output shapes, a hand-computed focal loss, heatmap targets decoded back to the same box, window coverage, patch padding, box and mask conversion to the original grid, all three tasks trained and scored on synthetic volumes, resume from a checkpoint and a stop at the deadline.

In [ ]:
x = torch.zeros(2, 1, 128, 64, 128)
for name, net in (("SEGNet", SEGNet()), ("DETNet", DETNet()), ("CLSNet", CLSNet())):
    with torch.no_grad():
        print(name, "output", tuple(net.eval()(x).shape), "parameters", f"{sum(q.numel() for q in net.parameters()):,}")
logit, heat = torch.tensor([0.0, 0.0]), torch.tensor([1.0, 0.5])
print("focal loss", round(p.heat_focal(logit, heat).item(), 5), "by hand", round((0.25 + 1 / 64) * np.log(2), 5))
boxes = torch.tensor([[20.0, 10.0, 30.0, 12.0, 6.0, 18.0]])
target, cells = p.heat_targets(boxes, (16, 8, 24))
offset = (boxes[:, :3].view(-1, 3, 1, 1, 1) - cells[None]) / 8
size = boxes[:, 3:].log().view(-1, 3, 1, 1, 1).expand_as(offset)
print("targets decode back to", p.decode(torch.cat([(target * 20 - 10)[:, None], offset, size], 1), k=1)[0][0, 0].numpy().round(3))

## 4. Images

**Method**: a volume goes to DETNet at 1 mm, which gives boxes; the top box centres a patch for CLSNet; SEGNet slides over the volume at 0.5 mm. The three models train separately, and CLSNet reads only DETNet's output. **Our outputs**: training curves below, once the run has finished.

In [ ]:
histories = sorted(RESULTS.glob("*_history.csv"))
if histories:
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.2))
    for a, task in zip(ax, ("det", "seg", "cls")):
        for s in range(3):
            h = pd.read_csv(RESULTS / f"luo2025_t1_{task}_seed{s}_history.csv")
            a.plot(h.epoch, h.val, label=f"seed {s}")
        a.set(title=task, xlabel="epoch", ylabel={"det": "top-box IoU", "seg": "Dice", "cls": "AUC"}[task]); a.legend(frameon=False)
    plt.show()
else:
    print("not run yet")

## 5. Tables

T1's published row and the leaderboard, with ours after the run. The challenge reports no seed variance. The paper has no ablations for T1.

### Challenge benchmark

Our method locates the lesion itself on every task, so it is comparable with the challenge entries, except that it was scored on the released Test labels rather than through the challenge's Docker system.

In [ ]:
display(PUBLISHED.loc[["T1 SZU"]])
if results.exists():
    display(pd.read_csv(results, index_col=0))
    display(pd.read_csv(RESULTS / "luo2025_t1_benchmark.csv", index_col=0).T)
else:
    print("not run yet")

## 6. Protocol

- **Split**: fit on Train (100), select on Validation (30), report on the official Test split (70), by volume.
- **Positive class**: malignant.
- **Seeds**: 3, with AUC bootstrap CI.
- **Re-run**: see [`docs/reproductions/luo2025-t1.md`](../docs/reproductions/luo2025-t1.md). Kernel in `kaggle/luo2025-t1/`.
- **Stored results**: `results/luo2025_t1/`, final weights and checkpoints on Drive in `abus-checkpoints/luo2025-t1`.

## 7. Open issues

- The paper does not say what T1's classifier reads or how it was trained, so a difference from AUC 0.889 cannot yet be attributed.
- Results are pending the Kaggle run.